# Amazon 2018 原始数据快速检查

In [ ]:
# 直接运行此单元即可同时检查 reviews 和 metadata。
# 如需缩小或扩大检查范围，只修改 INSPECT_LIMIT；不会读取完整大文件。

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import json

INSPECT_LIMIT = 1000
RAW_DIR = Path('D:/转码ing/MiniOneRec-main/data/raw/Amazon18')
INPUTS = [
    ('reviews', RAW_DIR / 'Industrial_and_Scientific.json'),
    ('metadata', RAW_DIR / 'meta_Industrial_and_Scientific.json'),
]

for file_kind, file_path in INPUTS:
    if not file_path.is_file():
        raise FileNotFoundError(f'找不到原始数据文件：{file_path}')

    print('\n' + '=' * 88)
    print(f'{file_kind.upper()}: {file_path}')
    print(f'文件大小：{file_path.stat().st_size / 1024 / 1024:.2f} MiB；检查上限：{INSPECT_LIMIT} 条')

    field_names = set()
    user_ids, item_ids = set(), set()
    rating_counts = Counter()
    records_checked = 0
    missing_review_text = 0
    missing_title = 0
    missing_description = 0
    earliest_timestamp, latest_timestamp = None, None

    # Amazon 2018 原始文件采用 JSON Lines 格式：每一行对应一条 JSON 记录。
    with file_path.open('r', encoding='utf-8') as raw_file:
        for line_number, line in enumerate(raw_file, start=1):
            if records_checked >= INSPECT_LIMIT:
                break
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                print(f'跳过第 {line_number} 行的无效 JSON：{error}')
                continue

            records_checked += 1
            field_names.update(record.keys())

            # 仅展示前 5 条样本；长评论和描述会被截断，避免大量输出原文。
            if records_checked <= 5:
                preview_keys = (
                    ('reviewerID', 'asin', 'overall', 'unixReviewTime', 'summary', 'reviewText')
                    if file_kind == 'reviews'
                    else ('asin', 'title', 'description', 'brand', 'category')
                )
                preview = {}
                for key in preview_keys:
                    if key not in record:
                        continue
                    text = str(record[key]).replace('\n', ' ').strip()
                    preview[key] = text if len(text) <= 180 else text[:180] + '...'
                print(f'样本 {records_checked}: ' + json.dumps(preview, ensure_ascii=False))

            asin = record.get('asin')
            if asin not in (None, ''):
                item_ids.add(str(asin))

            if file_kind == 'reviews':
                reviewer_id = record.get('reviewerID')
                if reviewer_id not in (None, ''):
                    user_ids.add(str(reviewer_id))
                rating = record.get('overall')
                rating_counts['missing' if rating in (None, '') else str(rating)] += 1
                if not str(record.get('reviewText') or '').strip():
                    missing_review_text += 1
                try:
                    timestamp = int(record.get('unixReviewTime'))
                    earliest_timestamp = timestamp if earliest_timestamp is None else min(earliest_timestamp, timestamp)
                    latest_timestamp = timestamp if latest_timestamp is None else max(latest_timestamp, timestamp)
                except (TypeError, ValueError):
                    pass
            else:
                if not str(record.get('title') or '').strip():
                    missing_title += 1
                description = record.get('description')
                if description is None or description == '' or description == []:
                    missing_description += 1

    print('\n--- 汇总 ---')
    print('字段名：', ', '.join(sorted(field_names)))
    print(f'已检查记录数：{records_checked}')
    print(f'样本内不同商品数：{len(item_ids)}')
    if file_kind == 'reviews':
        print(f'样本内不同用户数：{len(user_ids)}')
        print(f'样本内交互数：{records_checked}')
        print('评分分布：', dict(sorted(rating_counts.items())))
        print(f'reviewText 缺失：{missing_review_text}/{records_checked} ({missing_review_text / records_checked:.2%})' if records_checked else 'reviewText 缺失：无数据')
        if earliest_timestamp is not None:
            earliest_date = datetime.fromtimestamp(earliest_timestamp, tz=timezone.utc).isoformat()
            latest_date = datetime.fromtimestamp(latest_timestamp, tz=timezone.utc).isoformat()
            print(f'时间范围（UTC）：{earliest_date} 至 {latest_date}')
    else:
        print(f'title 缺失：{missing_title}/{records_checked} ({missing_title / records_checked:.2%})' if records_checked else 'title 缺失：无数据')
        print(f'description 缺失：{missing_description}/{records_checked} ({missing_description / records_checked:.2%})' if records_checked else 'description 缺失：无数据')



REVIEWS: D:\转码ing\MiniOneRec-main\data\raw\Amazon18\Industrial_and_Scientific.json
文件大小：740.17 MiB；检查上限：1000 条
样本 1: {"reviewerID": "A3FANY5GOT5X0W", "asin": "0176496920", "overall": "5.0", "unixReviewTime": "1358899200", "summary": "Just as described!", "reviewText": "Arrived on time, in mint condition, great!  I would buy something from this seller again in the future! Have a great day!"}
样本 2: {"reviewerID": "AT6HRPPYOPHMB", "asin": "0176496920", "overall": "5.0", "unixReviewTime": "1352073600", "summary": "Great device", "reviewText": "This device was hard to find for my daughter's OT school class from the description provided by the teacher. Fortunately, the key words were adequate to find the correct device, an..."}
样本 3: {"reviewerID": "A4IX7B38LIN1E", "asin": "0176496920", "overall": "4.0", "unixReviewTime": "1350432000", "summary": "Pretty Good", "reviewText": "Just a clicker nothing special. Was hoping it came in the box but just bubble wrap and a slip of paper. No complaint

In [4]:
# 检查文件大小
for name, path in INPUTS:
    count = 0
    with path.open("r", encoding="utf-8") as f:
        for _ in f:
            count += 1

    print(f"{name}: {count:,} records")

reviews: 1,758,333 records
metadata: 167,442 records


In [5]:
# 统计 Amazon 原始 reviews 文件中的交互数、不同用户数和不同商品数。
# 采用逐行读取，不会一次性加载完整文件到内存。

from pathlib import Path
import json

reviews_path = Path(
    r"D:\转码ing\MiniOneRec-main\data\raw\Amazon18\Industrial_and_Scientific.json"
)

user_ids = set()
item_ids = set()
interaction_count = 0

with reviews_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        record = json.loads(line)
        interaction_count += 1

        reviewer_id = record.get("reviewerID")
        asin = record.get("asin")

        if reviewer_id:
            user_ids.add(reviewer_id)
        if asin:
            item_ids.add(asin)

print("=== Reviews 原始数据统计 ===")
print(f"交互数: {interaction_count:,}")
print(f"不同用户数: {len(user_ids):,}")
print(f"不同商品数: {len(item_ids):,}")

=== Reviews 原始数据统计 ===
交互数: 1,758,333
不同用户数: 1,246,131
不同商品数: 165,764


In [6]:
# 统计 metadata 文件中的商品数，并检查与 reviews 商品集合的覆盖关系。

from pathlib import Path
import json

metadata_path = Path(
    r"D:\转码ing\MiniOneRec-main\data\raw\Amazon18\meta_Industrial_and_Scientific.json"
)

metadata_item_ids = set()

with metadata_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        record = json.loads(line)
        asin = record.get("asin")

        if asin:
            metadata_item_ids.add(asin)

print("=== Metadata 原始数据统计 ===")
print(f"不同商品数: {len(metadata_item_ids):,}")

review_items_without_metadata = item_ids - metadata_item_ids
metadata_items_without_reviews = metadata_item_ids - item_ids

print(f"reviews 中有、metadata 中没有的商品数: {len(review_items_without_metadata):,}")
print(f"metadata 中有、reviews 中没有的商品数: {len(metadata_items_without_reviews):,}")

=== Metadata 原始数据统计 ===
不同商品数: 165,687
reviews 中有、metadata 中没有的商品数: 82
metadata 中有、reviews 中没有的商品数: 5


如何理解“reviews 中有、metadata 中没有的商品数: 82、metadata 中有、reviews 中没有的商品数: 5”

后续用于构建商品 embedding 的商品，应该理解为 ***reviews 与 metadata 能通过 asin 对上的商品，并且还要通过预处理中的标题与 k-core 筛选***。

reviews提供用户—商品交互序列

metadata提供商品 title、description -> item embedding

没有 metadata 的商品没有标题/描述，无法按当前流程生成可靠的文本 embedding；没有 reviews 的商品虽可生成 embedding，但没有用户交互，通常不会进入推荐训练样本。